# Advanced Analysis — Questions 21-30

**Techniques:** `RFM` · Cohort analysis · Gaps & Islands · Duplicate detection · YoY growth · SQL Pivot · Customer Lifetime Value

| # | Question |
|---|----------|
| Q21 | RFM customer segmentation |
| Q22 | Cohort analysis — monthly retention |
| Q23 | Late delivery impact on review scores (+ t-test) |
| Q24 | Seller composite performance score |
| Q25 | Month-over-month revenue growth by category |
| Q26 | Duplicate detection and deduplication |
| Q27 | Gaps & Islands — churned and reactivated customers |
| Q28 | Year-over-year revenue growth by category |
| Q29 | SQL Pivot — payment method breakdown by month |
| Q30 | Customer Lifetime Value (CLV) scoring |

In [ ]:
import sys
sys.path.append('../')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from src.db_utils import run_query
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## Q21 — RFM Customer Segmentation
`CTE` chain · `NTILE()` · `CASE WHEN`

In [ ]:
q21 = run_query("""
    WITH rfm_base AS (
        SELECT c.customer_unique_id,
               MAX(o.order_purchase_timestamp) AS last_purchase,
               COUNT(DISTINCT o.order_id) AS frequency,
               ROUND(SUM(op.payment_value), 2) AS monetary
        FROM customers c JOIN orders o ON c.customer_id = o.customer_id
        JOIN order_payments op ON o.order_id = op.order_id
        WHERE o.order_status = 'delivered' GROUP BY c.customer_unique_id
    ),
    rfm_scores AS (
        SELECT *, NTILE(5) OVER (ORDER BY last_purchase DESC) AS r_score,
               NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
               NTILE(5) OVER (ORDER BY monetary ASC) AS m_score
        FROM rfm_base
    )
    SELECT *, (r_score + f_score + m_score) AS rfm_total,
           CASE WHEN (r_score+f_score+m_score) >= 13 THEN 'Champions'
                WHEN (r_score+f_score+m_score) >= 10 THEN 'Loyal Customers'
                WHEN (r_score+f_score+m_score) >= 7  THEN 'Potential Loyalists'
                WHEN (r_score+f_score+m_score) >= 4  THEN 'At Risk'
                ELSE 'Lost' END AS segment
    FROM rfm_scores ORDER BY rfm_total DESC
""")
seg = q21.groupby('segment').agg(count=('customer_unique_id','count'), avg_spend=('monetary','mean')).round(2).reset_index()
order = ['Champions','Loyal Customers','Potential Loyalists','At Risk','Lost']
colors_map = {'Champions':'#2ecc71','Loyal Customers':'#3498db','Potential Loyalists':'#f39c12','At Risk':'#e67e22','Lost':'#e74c3c'}
seg_s = seg.set_index('segment').reindex(order).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
bar_c = [colors_map[s] for s in seg_s['segment']]
axes[0].bar(seg_s['segment'], seg_s['count'], color=bar_c)
axes[0].set_title('Customer Count by Segment')
axes[0].set_xticklabels(seg_s['segment'], rotation=20, ha='right')
axes[1].bar(seg_s['segment'], seg_s['avg_spend'], color=bar_c)
axes[1].set_title('Avg Spending by Segment (BRL)')
axes[1].set_xticklabels(seg_s['segment'], rotation=20, ha='right')
plt.suptitle('Q21 — RFM Customer Segmentation', fontweight='bold')
plt.tight_layout(); plt.savefig('../images/q21_rfm_segmentation.png', bbox_inches='tight'); plt.show()
seg_s

## Q22 — Cohort Analysis: Monthly Retention
`CTE` · `JOIN` · Heatmap visualization

In [ ]:
q22 = run_query("""
    WITH cb AS (
        SELECT c.customer_unique_id,
               STRFTIME('%Y-%m', MIN(o.order_purchase_timestamp)) AS cohort_month,
               STRFTIME('%Y-%m', o.order_purchase_timestamp) AS order_month
        FROM customers c JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.order_status = 'delivered'
        GROUP BY c.customer_unique_id, order_month
    ),
    cs AS (SELECT cohort_month, COUNT(DISTINCT customer_unique_id) AS cohort_n FROM cb GROUP BY cohort_month),
    ret AS (SELECT cohort_month, order_month, COUNT(DISTINCT customer_unique_id) AS active FROM cb GROUP BY cohort_month, order_month)
    SELECT r.cohort_month, r.order_month, cs.cohort_n, r.active,
           ROUND(r.active * 100.0 / cs.cohort_n, 2) AS retention_rate
    FROM ret r JOIN cs ON r.cohort_month = cs.cohort_month
    ORDER BY r.cohort_month, r.order_month
""")
pivot = q22.pivot_table(index='cohort_month', columns='order_month', values='retention_rate', aggfunc='first')
plt.figure(figsize=(16, 8))
sns.heatmap(pivot.iloc[:12, :12], annot=True, fmt='.1f', cmap='YlOrRd_r',
            linewidths=0.5, cbar_kws={'label': 'Retention Rate (%)'})
plt.title('Q22 — Cohort Retention Heatmap (%)', fontweight='bold')
plt.xlabel('Order Month'); plt.ylabel('Cohort Month')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.savefig('../images/q22_cohort_retention.png', bbox_inches='tight'); plt.show()
print(f'Cohorts analyzed: {q22["cohort_month"].nunique()}')

## Q23 — Late Delivery Impact on Review Scores
`CASE WHEN` · Python Welch's t-test for statistical significance

In [ ]:
q23 = run_query("""
    WITH ds AS (
        SELECT CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                    THEN 'Late' ELSE 'On Time' END AS flag,
               r.review_score
        FROM orders o JOIN order_reviews r ON o.order_id = r.order_id
        WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
    )
    SELECT flag, COUNT(*) AS total, ROUND(AVG(review_score), 3) AS avg_score,
           SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) AS low_scores,
           ROUND(SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS low_pct
    FROM ds GROUP BY flag
""")
raw = run_query("""
    SELECT CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN 'Late' ELSE 'On Time' END AS flag,
           CAST(r.review_score AS FLOAT) AS review_score
    FROM orders o JOIN order_reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
""")
t_stat, p_value = stats.ttest_ind(
    raw[raw['flag'] == 'On Time']['review_score'],
    raw[raw['flag'] == 'Late']['review_score'], equal_var=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
palette = {'On Time': '#2ecc71', 'Late': '#e74c3c'}
sns.barplot(data=q23, x='flag', y='avg_score', ax=axes[0], hue='flag', palette=palette, legend=False)
axes[0].set_title('Avg Review Score'); axes[0].set_ylim(0, 5)
sns.barplot(data=q23, x='flag', y='low_pct', ax=axes[1], hue='flag', palette=palette, legend=False)
axes[1].set_title('% Orders with Score <= 2')
sig = 'SIGNIFICANT' if p_value < 0.05 else 'NOT significant'
plt.suptitle(f'Q23 — Late Delivery Impact (p={p_value:.2e}, {sig})', fontweight='bold')
plt.tight_layout(); plt.savefig('../images/q23_delivery_vs_score.png', bbox_inches='tight'); plt.show()
q23

## Q24 — Seller Composite Performance Score
`NTILE()` · weighted scoring formula

In [ ]:
q24 = run_query("""
    WITH ss AS (
        SELECT oi.seller_id, COUNT(DISTINCT oi.order_id) AS orders,
               ROUND(SUM(oi.price), 2) AS revenue,
               ROUND(AVG(r.review_score), 3) AS avg_review,
               ROUND(AVG(JULIANDAY(o.order_delivered_customer_date)-JULIANDAY(o.order_purchase_timestamp)),1) AS avg_days
        FROM order_items oi JOIN orders o ON oi.order_id = o.order_id
        JOIN order_reviews r ON o.order_id = r.order_id
        WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
        GROUP BY oi.seller_id HAVING orders >= 10
    ),
    sc AS (
        SELECT *, NTILE(5) OVER (ORDER BY revenue ASC) AS vs,
               NTILE(5) OVER (ORDER BY avg_review ASC) AS rs,
               NTILE(5) OVER (ORDER BY avg_days DESC) AS ss2
        FROM ss
    )
    SELECT seller_id, orders, revenue, avg_review, avg_days,
           ROUND(vs*0.4 + rs*0.4 + ss2*0.2, 2) AS composite
    FROM sc ORDER BY composite DESC LIMIT 20
""")
fig, ax = plt.subplots(figsize=(10, 6))
c_vals = q24['composite'].values
ax.barh(range(len(q24)), c_vals, color=plt.cm.RdYlGn(c_vals / 5))
ax.set_yticks(range(len(q24)))
ax.set_yticklabels([s[:14] + '...' for s in q24['seller_id']], fontsize=8)
ax.set_xlabel('Composite Score (max 5.0)')
ax.set_title('Q24 — Top 20 Sellers: Composite Score\n(40% Volume + 40% Review + 20% Speed)', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout(); plt.savefig('../images/q24_seller_scores.png', bbox_inches='tight'); plt.show()
q24.head(10)

## Q25 — Month-over-Month Revenue Growth by Category
`LAG()` · `PARTITION BY` · `NULLIF`

In [ ]:
q25 = run_query("""
    WITH mcr AS (
        SELECT STRFTIME('%Y-%m', o.order_purchase_timestamp) AS month,
               t.product_category_name_english AS category,
               ROUND(SUM(oi.price), 2) AS revenue
        FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
        JOIN products p ON oi.product_id = p.product_id
        JOIN product_category_name_translation t ON p.product_category_name = t.product_category_name
        WHERE o.order_status = 'delivered'
        GROUP BY month, category
    ),
    wl AS (
        SELECT month, category, revenue,
               LAG(revenue) OVER (PARTITION BY category ORDER BY month) AS prev
        FROM mcr
    )
    SELECT month, category, revenue, prev,
           ROUND((revenue - prev) * 100.0 / NULLIF(prev, 0), 2) AS mom_pct
    FROM wl WHERE prev IS NOT NULL
    ORDER BY mom_pct DESC LIMIT 30
""")
top = q25.head(10)
fig, ax = plt.subplots(figsize=(11, 6))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in top['mom_pct']]
labels = [f"{r['category']} ({r['month']})" for _, r in top.iterrows()]
ax.barh(labels, top['mom_pct'], color=colors)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('MoM Growth (%)')
ax.set_title('Q25 — Top 10 Category-Months by MoM Revenue Growth', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout(); plt.savefig('../images/q25_mom_growth.png', bbox_inches='tight'); plt.show()
q25.head(10)

## Q26 — Duplicate Detection and Deduplication
`ROW_NUMBER()` — most common data quality interview pattern

In [ ]:
# Step 1: Find orders with more than one review
q26_dupes = run_query("""
    SELECT order_id, COUNT(*) AS review_count
    FROM order_reviews GROUP BY order_id
    HAVING review_count > 1
    ORDER BY review_count DESC LIMIT 10
""")
print(f'Orders with duplicate reviews: {len(q26_dupes)}')
print(q26_dupes)

# Step 2: Deduplicate — keep only the most recent review per order
q26_clean = run_query("""
    WITH deduped AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY review_creation_date DESC) AS rn
        FROM order_reviews
    )
    SELECT order_id, review_id, review_score, review_creation_date
    FROM deduped WHERE rn = 1
    LIMIT 10
""")
print('\nAfter deduplication (1 review per order):')
q26_clean

## Q27 — Gaps & Islands: Customers Who Churned and Came Back
`LAG()` · `JULIANDAY()` — classic gap detection pattern

In [ ]:
q27 = run_query("""
    WITH customer_orders AS (
        SELECT c.customer_unique_id, o.order_purchase_timestamp AS order_date,
               LAG(o.order_purchase_timestamp) OVER (
                   PARTITION BY c.customer_unique_id ORDER BY o.order_purchase_timestamp
               ) AS prev_order_date
        FROM customers c JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.order_status = 'delivered'
    ),
    gaps AS (
        SELECT customer_unique_id, prev_order_date AS churned_after, order_date AS returned_on,
               ROUND(JULIANDAY(order_date) - JULIANDAY(prev_order_date), 0) AS gap_days
        FROM customer_orders WHERE prev_order_date IS NOT NULL
    )
    SELECT * FROM gaps WHERE gap_days > 60 ORDER BY gap_days DESC LIMIT 20
""")
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(run_query("""
    WITH co AS (
        SELECT c.customer_unique_id, o.order_purchase_timestamp AS od,
               LAG(o.order_purchase_timestamp) OVER (PARTITION BY c.customer_unique_id ORDER BY o.order_purchase_timestamp) AS prev
        FROM customers c JOIN orders o ON c.customer_id = o.customer_id WHERE o.order_status = 'delivered'
    )
    SELECT ROUND(JULIANDAY(od) - JULIANDAY(prev), 0) AS gap_days
    FROM co WHERE prev IS NOT NULL AND JULIANDAY(od) - JULIANDAY(prev) > 0
""")['gap_days'], bins=50, color='steelblue', edgecolor='white')
ax.axvline(60, color='red', linestyle='--', linewidth=1.5, label='60-day churn threshold')
ax.set_title('Q27 — Distribution of Order Gaps (Days Between Purchases)', fontweight='bold')
ax.set_xlabel('Gap (days)'); ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout(); plt.savefig('../images/q27_order_gaps.png', bbox_inches='tight'); plt.show()
print(f'Customers with 60+ day gap (churned & returned): {len(q27)}')
q27.head(10)

## Q28 — Year-over-Year Revenue Growth by Category
`LAG() OVER (PARTITION BY)` — YoY pattern

In [ ]:
q28 = run_query("""
    WITH yearly AS (
        SELECT STRFTIME('%Y', o.order_purchase_timestamp) AS year,
               t.product_category_name_english AS category,
               ROUND(SUM(oi.price), 2) AS revenue
        FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
        JOIN products p ON oi.product_id = p.product_id
        JOIN product_category_name_translation t ON p.product_category_name = t.product_category_name
        WHERE o.order_status = 'delivered'
        GROUP BY year, category
    )
    SELECT category, year, revenue,
           LAG(revenue) OVER (PARTITION BY category ORDER BY year) AS prev_year,
           ROUND((revenue - LAG(revenue) OVER (PARTITION BY category ORDER BY year)) * 100.0
                 / NULLIF(LAG(revenue) OVER (PARTITION BY category ORDER BY year), 0), 2) AS yoy_pct
    FROM yearly WHERE year IN ('2017', '2018')
    ORDER BY yoy_pct DESC LIMIT 20
""")
fig, ax = plt.subplots(figsize=(11, 6))
top_yoy = q28.dropna().head(15)
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in top_yoy['yoy_pct']]
ax.barh(top_yoy['category'], top_yoy['yoy_pct'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('YoY Growth (%)')
ax.set_title('Q28 — Year-over-Year Revenue Growth by Category (2017 vs 2018)', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout(); plt.savefig('../images/q28_yoy_growth.png', bbox_inches='tight'); plt.show()
q28.dropna().head(10)

## Q29 — SQL Pivot: Payment Method Breakdown by Month
Conditional aggregation · `CASE WHEN` inside `COUNT()`

In [ ]:
q29 = run_query("""
    SELECT STRFTIME('%Y-%m', o.order_purchase_timestamp) AS month,
           COUNT(CASE WHEN op.payment_type = 'credit_card' THEN 1 END) AS credit_card,
           COUNT(CASE WHEN op.payment_type = 'boleto'      THEN 1 END) AS boleto,
           COUNT(CASE WHEN op.payment_type = 'voucher'     THEN 1 END) AS voucher,
           COUNT(CASE WHEN op.payment_type = 'debit_card'  THEN 1 END) AS debit_card,
           COUNT(*) AS total
    FROM orders o JOIN order_payments op ON o.order_id = op.order_id
    GROUP BY month ORDER BY month
""")
fig, ax = plt.subplots(figsize=(14, 6))
cols = ['credit_card', 'boleto', 'voucher', 'debit_card']
colors = ['#3498db', '#e67e22', '#2ecc71', '#9b59b6']
bottom = [0] * len(q29)
for col, color in zip(cols, colors):
    ax.bar(range(len(q29)), q29[col], bottom=bottom, label=col.replace('_', ' ').title(), color=color)
    bottom = [b + v for b, v in zip(bottom, q29[col])]
ax.set_xticks(range(len(q29)))
ax.set_xticklabels(q29['month'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Order Count')
ax.set_title('Q29 — Payment Method Breakdown by Month (SQL Pivot)', fontweight='bold')
ax.legend(loc='upper left')
plt.tight_layout(); plt.savefig('../images/q29_payment_pivot.png', bbox_inches='tight'); plt.show()
q29.tail(10)

## Q30 — Customer Lifetime Value (CLV) Scoring
Multi-CTE · recency weighting · `NTILE()` quartiles

In [ ]:
q30 = run_query("""
    WITH customer_metrics AS (
        SELECT c.customer_unique_id,
               COUNT(DISTINCT o.order_id) AS total_orders,
               ROUND(SUM(op.payment_value), 2) AS total_spend,
               MIN(o.order_purchase_timestamp) AS first_order,
               MAX(o.order_purchase_timestamp) AS last_order,
               ROUND(JULIANDAY(MAX(o.order_purchase_timestamp)) - JULIANDAY(MIN(o.order_purchase_timestamp)), 0) AS lifespan_days,
               ROUND(JULIANDAY('2018-10-17') - JULIANDAY(MAX(o.order_purchase_timestamp)), 0) AS days_since_last
        FROM customers c JOIN orders o ON c.customer_id = o.customer_id
        JOIN order_payments op ON o.order_id = op.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY c.customer_unique_id HAVING total_orders >= 2
    ),
    clv AS (
        SELECT *,
               ROUND(total_spend / NULLIF(lifespan_days, 0) * 365, 2) AS annual_value,
               ROUND(total_spend * (1.0 / (1 + days_since_last / 365.0)), 2) AS clv_score
        FROM customer_metrics
    )
    SELECT customer_unique_id, total_orders, total_spend, lifespan_days,
           annual_value, clv_score,
           NTILE(4) OVER (ORDER BY clv_score DESC) AS clv_quartile
    FROM clv ORDER BY clv_score DESC LIMIT 30
""")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
quartile_data = run_query("""
    WITH cm AS (
        SELECT c.customer_unique_id,
               COUNT(DISTINCT o.order_id) AS total_orders,
               ROUND(SUM(op.payment_value), 2) AS total_spend,
               ROUND(JULIANDAY(MAX(o.order_purchase_timestamp)) - JULIANDAY(MIN(o.order_purchase_timestamp)), 0) AS lifespan_days,
               ROUND(JULIANDAY('2018-10-17') - JULIANDAY(MAX(o.order_purchase_timestamp)), 0) AS days_since_last
        FROM customers c JOIN orders o ON c.customer_id = o.customer_id
        JOIN order_payments op ON o.order_id = op.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY c.customer_unique_id HAVING total_orders >= 2
    ),
    clv AS (SELECT *, ROUND(total_spend * (1.0 / (1 + days_since_last / 365.0)), 2) AS clv_score FROM cm)
    SELECT NTILE(4) OVER (ORDER BY clv_score DESC) AS quartile, AVG(clv_score) AS avg_clv, AVG(total_spend) AS avg_spend
    FROM clv GROUP BY quartile
""")
colors_q = ['#2ecc71','#3498db','#f39c12','#e74c3c']
axes[0].bar([f'Q{int(q)}' for q in quartile_data['quartile']], quartile_data['avg_clv'], color=colors_q)
axes[0].set_title('Avg CLV Score by Quartile'); axes[0].set_ylabel('CLV Score')
axes[1].bar([f'Q{int(q)}' for q in quartile_data['quartile']], quartile_data['avg_spend'], color=colors_q)
axes[1].set_title('Avg Total Spend by Quartile (BRL)'); axes[1].set_ylabel('Avg Spend (BRL)')
plt.suptitle('Q30 — Customer Lifetime Value Quartiles', fontweight='bold')
plt.tight_layout(); plt.savefig('../images/q30_clv_quartiles.png', bbox_inches='tight'); plt.show()
print(f'High-value customers (Q1): {int(len(q30)/4)} identified')
q30.head(10)